In [1]:
import os
import sys
file_path = os.getcwd()
project_root = os.path.dirname(file_path)
print(project_root)
# Add the project root to the sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
from dotenv import load_dotenv
import chromadb
from langchain.agents import AgentExecutor
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma 
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document
from pathlib import Path
from langchain import hub
from langchain_core.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import json
# Our Libraries
from crunch_llm.code.agents import *
from crunch_llm.code.utils import *
from typing import Optional

/oscar/data/gk/jdtoscan/LLM


/users/jdtoscan/.conda/envs/Agent_Hub/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3699: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  exec(code_obj, self.user_global_ns, self.user_ns)
/users/jdtoscan/.conda/envs/Agent_Hub/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def initialize_local_data(db_path: str, collection_name='documents', model='bge-m3:567m'):
    """
    Initializes or loads a local Chroma vector store using an explicit client.
    """
    # 1. Create an explicit PersistentClient. This is more robust.
    client = chromadb.PersistentClient(path=db_path)
    
    # 2. Initiate embeddings model (same as before)
    embeddings = OllamaEmbeddings(model=model)
    
    # 3. Pass the explicit client to the LangChain Chroma wrapper
    vector_store = Chroma(
        client=client, # Pass the client object here
        collection_name=collection_name,
        embedding_function=embeddings,
    )
    # Note: We no longer need 'persist_directory' because the client handles it.
    
    print(f"✅ Vector store initialized from path: {db_path}")
    
    return vector_store

# Initialization

In [3]:
analyzer_system_prompt = """
You are a senior researcher in computational science. Your task is to analyze the provided code snippet and extract key information into a structured JSON format.

- If the code IS a PDE solver: Identify the 'equation', 'method', 'boundary_conditions', and 'problem_type'.
- If the code is NOT a PDE solver: Fill the specialized fields with a null value and provide a general 'description'.

Your final output must be a single, valid JSON object. Do not add any explanatory text.
"""

In [4]:
splitter_system_prompt = """
You are an expert programmer who specializes in refactoring and understanding code structure. Your sole job is to identify the most logical line numbers to split the following code for semantic chunking.

Focus on breaking the code at the end of functions, logical blocks (like setup, solving, plotting), or major comments.

Your final output must be a single, valid JSON object with a single key, "split_points", containing a list of integers. Do not add any other text, explanation, or keys.
"""

# Iterate over files:

In [5]:
# 1. Define the exact path where you want to save the database
my_database_path = "/users/jdtoscan/data/jdtoscan/LLM/Documents/database_pinns"

# 2. Pass that path to your function
vector_store = initialize_local_data(db_path=my_database_path)

✅ Vector store initialized from path: /users/jdtoscan/data/jdtoscan/LLM/Documents/database_pinns


In [6]:
analyzer_agent = BasicStructuredAgent(
                     model='gpt-oss:120b', 
                     system_prompt=analyzer_system_prompt,
                     verbose=False)         

splitter_agent = BasicStructuredAgent(
                     model='gpt-oss:120b', 
                     system_prompt=splitter_system_prompt,
                     verbose=False)                     

# Generate Metadata and Ingest

In [7]:
def create_smart_chunks(good_points: list[int], total_lines: int, 
                        target_size: int = 90, overlap_lines: int = 0) -> list[tuple[int, int]]:
    """
    Takes a list of semantically important line numbers ("good points") and creates
    larger, overlapping chunks that snap to these points.

    Args:
        good_points (list[int]): List of line numbers identified by the agent.
        total_lines (int): The total number of lines in the code.
        target_size (int): The desired size for each chunk.
        overlap_lines (int): The desired number of lines for overlap.

    Returns:
        A list of (start_line, end_line) tuples.
    """
    if not good_points:
        return [(0, total_lines)] # If no points, return the whole document as one chunk

    # Ensure points are sorted and unique, and include the start and end
    points = sorted(list(set([0] + good_points + [total_lines])))
    
    chunks = []
    current_line = 0

    while current_line < total_lines:
        # Find the best end point for the current chunk
        # It should be a good_point around (current_line + target_size)
        ideal_end = current_line + target_size
        
        # Find the closest available point in our list that is >= ideal_end
        # If none, just take the last point.
        best_end = next((p for p in points if p >= ideal_end), points[-1])
        
        # To avoid tiny final chunks, if the last chunk is too small, merge it
        if best_end > current_line and (total_lines - best_end) < (target_size / 2):
            best_end = total_lines
            
        chunks.append((current_line, best_end))
        
        # If we've reached the end, stop
        if best_end == total_lines:
            break
            
        # Determine the start of the next chunk for overlap
        ideal_next_start = best_end - overlap_lines
        # Find the closest available point that is <= ideal_next_start
        current_line = next((p for p in reversed(points) if p <= ideal_next_start and p > current_line), best_end)

    return chunks

In [8]:
import json
from pathlib import Path
from langchain_core.documents import Document

def ingest_with_dedicated_splitter(file_path, code_content, analysis_metadata, split_points, vector_store,
                                   min_lines_for_split=100, target_chunk_size=70, overlap=0):
    """
    Ingests code by creating smart chunks and prepending a formatted metadata
    header to the content of each chunk.
    """
    
    # --- Part 1: Prepare Metadata and Language-Specific Info ---
    base_metadata = {
        'source_file': file_path,
        'filename': Path(file_path).name,
        'type': 'code',
        **analysis_metadata
    }

    # Determine language and comment style for the header
    file_ext = Path(file_path).suffix.lower()
    language_map = {'.py': 'python', '.m': 'matlab', '.jl': 'julia', '.cpp': 'cpp', '.c': 'c'}
    language = language_map.get(file_ext, 'unknown')
    comment_styles = {'python': '#', 'matlab': '%', 'julia': '#', 'c': '//', 'cpp': '//'}
    comment = comment_styles.get(language, '#')
    
    # Format the agent's analysis into a clean string for the header
    analysis_str = json.dumps(analysis_metadata, indent=2).replace('{', '').replace('}', '').replace('"', '')

    code_lines = code_content.splitlines()
    total_lines = len(code_lines)
    docs_to_add = []

    # --- Part 2: Python Control Logic with Header Prepending ---
    if total_lines > min_lines_for_split and split_points:
        print(f"  → File is long ({total_lines} lines). Applying smart chunking...")
        chunk_boundaries = create_smart_chunks(split_points, total_lines, target_chunk_size, overlap)
        print(f"  → Generated {len(chunk_boundaries)} smart chunks.")

        for i, (start_line, end_line) in enumerate(chunk_boundaries):
            chunk_content = "\n".join(code_lines[start_line:end_line])
            chunk_metadata = base_metadata.copy()
            chunk_metadata['lines'] = f"{start_line}-{end_line}"
            chunk_metadata['part'] = f"{i+1}/{len(chunk_boundaries)}"
            
            # ✨ CREATE THE HEADER for this specific chunk ✨
            header = f"""{comment} CODE ANALYSIS (PART {i+1} of {len(chunk_boundaries)})
{comment} {analysis_str.replace(chr(10), f'{chr(10)}{comment} ')}
{comment} =======================================================\n\n"""
            
            print(header)
            enhanced_content = header + chunk_content
            doc = Document(page_content=enhanced_content, metadata=chunk_metadata)
            docs_to_add.append(doc)
    else:
        # If the file is short, add a header without the part number
        print(f"  → File is short ({total_lines} lines). Storing as a single chunk.")
        
        # ✨ CREATE THE HEADER for the single document ✨
        header = f"""{comment} CODE ANALYSIS
{comment} {analysis_str.replace(chr(10), f'{chr(10)}{comment} ')}
{comment} =======================================================\n\n"""
        print(header)
        enhanced_content = header + code_content
        doc = Document(page_content=enhanced_content, metadata=base_metadata)
        docs_to_add.append(doc)

    # --- Part 3: Add to Vector Store ---
    if docs_to_add:
        vector_store.add_documents(docs_to_add)

In [9]:
# The directory containing your code files
target_path = '/users/jdtoscan/data/jdtoscan/LLM/Documents/codes/SciML/PINNs/'

all_results = {}
print(f"\nStarting analysis of all files in: {target_path}")

# os.walk will traverse the entire directory tree
for root, dirs, files in os.walk(target_path):
    for filename in files:
        file_path = os.path.join(root, filename)
        
        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                code_content = f.read()
            line_count = code_content.count('\n') + 1
            print(f"Processing: {filename} ({line_count} lines)...")
            
            # --- Run the two-agent pipeline ---
            print("  - Running Analyzer Agent...")
            analysis_result = analyzer_agent.run(code_content)
            
            print("  - Running Splitter Agent...")
            split_result = splitter_agent.run(code_content)
            split_points = split_result.get('split_points', []) # Safely get the list
            print(split_points)
            # Store the combined results if needed
            all_results[file_path] = {
                "analysis": analysis_result,
                "suggested_splits": split_points
            }
            
            print(f"Ingesting: {filename}...")
            # --- Call the NEW ingestion function ---
            
            ingest_with_dedicated_splitter(
                file_path, code_content, 
                analysis_result, split_points, # Pass both results
                vector_store
            )
            print(f"✅ Ingested: {filename}")
            
        except Exception as e:
            print(f"--- UNHANDLED ERROR analyzing {filename}: {e} ---")
            all_results[file_path] = {"error": str(e)}

# --- 4. Save and Display Results ---

output_file = 'full_code_analysis.json'
print(f"\nAnalysis complete! ✅ Saving all results to {output_file}...")

# Save the collected results to a single, comprehensive JSON file
with open(output_file, 'w') as f:
    json.dump(all_results, f, indent=4)

print("Done.")


Starting analysis of all files in: /users/jdtoscan/data/jdtoscan/LLM/Documents/codes/SciML/PINNs/
Processing: AllenCahn_PINN.py (811 lines)...
  - Running Analyzer Agent...
  - Running Splitter Agent...
--- UNHANDLED ERROR analyzing AllenCahn_PINN.py: Invalid json output: ```python
#!/usr/bin/env python3
"""
csv_column_sum.py – Sum numeric values from a named column in a CSV file.

Usage
-----
    python csv_column_sum.py <path/to/file.csv> <column_name>

Positional arguments
--------------------
    file_path      Path to the CSV file to be processed.
    column_name    Name of the column whose values should be summed.

Behavior
--------
* The script reads the CSV file using :mod:`csv.DictReader`.
* It verifies that the requested column exists; otherwise it exits with an
  informative error message.
* Each row is examined; values that can be converted to ``float`` are added to
  the running total.  Non‑numeric values are ignored and a warning is printed
  to ``stderr``.
* After proce

## Read and Delete Files

In [26]:
# Assuming 'vector_store' is your initialized Chroma object
print("Fetching all documents from the vector store...")

# The .get() method returns a dictionary with IDs, documents, metadatas, etc.
# We specify what information we want to retrieve.
all_entries = vector_store.get(
    include=["metadatas", "documents"] # You can also include "embeddings"
)

# Let's print the first 5 documents found
num_entries = len(all_entries['ids'])
print(f"Found {num_entries} total entries.\n")

for i in range(min(5, num_entries)):
    doc_id = all_entries['ids'][i]
    metadata = all_entries['metadatas'][i]
    content = all_entries['documents'][i]
    
    print(f"--- Entry {i+1} ---")
    print(f"ID: {doc_id}")
    print(f"Source File: {metadata.get('source_file', 'N/A')}")
    print(f"Content Preview: {content[:150]}...") # Print first 150 chars
    print("-" * 20)

Fetching all documents from the vector store...
Found 5 total entries.

--- Entry 1 ---
ID: c583cb39-dfa1-4525-8bad-35a3fe6876a8
Source File: /users/jdtoscan/data/jdtoscan/LLM/Documents/codes/Numerical_Methods/Burgers/julia/Burgers.jl
Content Preview: # CODE ANALYSIS (PART 1 of 2)
# 
#   equation: Viscous Burgers' equation: u_t + u u_x = \u03bd u_xx,
#   method: Fourier spectral discretization with ...
--------------------
--- Entry 2 ---
ID: 9e840975-8f05-4156-8b45-9dd9af3670ed
Source File: /users/jdtoscan/data/jdtoscan/LLM/Documents/codes/Numerical_Methods/Burgers/julia/Burgers.jl
Content Preview: # CODE ANALYSIS (PART 2 of 2)
# 
#   equation: Viscous Burgers' equation: u_t + u u_x = \u03bd u_xx,
#   method: Fourier spectral discretization with ...
--------------------
--- Entry 3 ---
ID: 5293549c-2cea-4527-8d8d-651eac09ee93
Source File: /users/jdtoscan/data/jdtoscan/LLM/Documents/codes/Numerical_Methods/Burgers/julia/Burgers_DG.jl
Content Preview: # CODE ANALYSIS
# 
#   equation: Vi

In [21]:
# # Assuming 'vector_store' is your initialized Chroma object

# # 1. Fetch BOTH ids and metadatas from the database.
# response = vector_store.get(include=["metadatas"]) # The 'ids' are returned by default
# all_ids = response['ids']
# all_metadatas = response['metadatas']

# # 2. Use zip to iterate over ids and metadatas simultaneously
# target_directory = "/Numerical_Methods/"
# ids_to_delete = []

# # zip() pairs each id with its corresponding metadata dictionary
# for doc_id, metadata in zip(all_ids, all_metadatas):
#     # Use Python's 'in' operator for a flexible substring search
#     if target_directory in metadata.get('source_file', ''):
#         ids_to_delete.append(doc_id) # Append the correct ID

# # Now 'ids_to_delete' contains the correct list of document IDs
# print(f"Found {len(ids_to_delete)} entries from the directory '{target_directory}' to delete.")

# # 3. Proceed with the deletion
# if ids_to_delete:
#     print(f"Deleting {len(ids_to_delete)} entries...")
#     vector_store.delete(ids=ids_to_delete)
#     print("✅ Deletion complete.")

Found 4 entries from the directory '/Numerical_Methods/' to delete.
Deleting 4 entries...
✅ Deletion complete.
